> Task2：  
> - 问题的定义：调用标准库 SB3 ，解决稍微困难些的问题 PandaReach-v3
> - 目标：  
> 1. 尝试调用 SB3 ，了解其功能   
> 2. 尝试调用 SB3 处理一个未知环境（例如 PandaReach-v3），并了解 SOP 流程 。如何探索环境？如何调用 SB3 选择算法？如何逐步调试到最终落实？等等  

PandaReach-v3：  
- Task：Reach。机械臂需要移动其末端执行器（End-effector，即“手掌”位置），去触碰空间中随机生成的一个红色目标点（Target）。  
- Challenge：机械臂需要学习三维空间的坐标转换和运动规划，以最短路径到达目标。  
- Robot：Franka Emika Panda。7DOF（旋转关节）。  
- 仿真引擎：基于 PyBullet，提供真实的重力、摩擦力及碰撞物理效果  
- Action Space：3维 (末端执行器X, Y, Z 位移)。Box(-1.0, 1.0, (3,))，连续动作空间。PS：3D 位移指令会通过环境内部的 逆运动学 (Inverse Kinematics, IK) 自动转换为 7 个关节的旋转角度。  
- Observation Space：12 (6 obs + 3 achieved + 3 desired)  
- Reward Setting：选择Dense Reward。末端执行器与目标点之间 欧几里得距离的负值  
- done 条件：Success：当末端执行器距离目标的欧氏距离 ≤0.05m (5厘米) 时，认为任务成功。Truncated：50 步内没能到达目标，该回合结束。  

## Part1: 探索环境

> PS: 关于物理引擎：通过 gymnasium 接口统一调用 mujoco或者 pybullet。  

> import:  
mujoco：import gymnasium as gym  
pybullet：import gymnasium as gym+import panda_gym  

> 渲染方式：  
gym.make(env_id, render_mode="human") -> 弹出一个物理引擎的 3D 窗口。  
gym.make(env_id, render_mode="rgb_array") -> 不弹窗，只返回图像数据（用于记录视频）。

> 注意observation：  
Panda-Gym (PyBullet)：大多数环境是 Goal-Conditioned (目标导向) 的。它们返回的 observation 是一个 字典 (Dict)，包含 observation、achieved_goal 和 desired_goal。  
你的代码中 SB3 会自动识别这种字典并使用 MultiInputPolicy（如果你没显式指定的话）。  
标准 MuJoCo (如 Ant-v4)：大多数返回的是一个简单的 数组 (Box/Array)。  
这时候 SB3 需要使用 MlpPolicy。  

In [1]:
import gymnasium as gym
import panda_gym

env = gym.make("PandaReach-v3")

print(f"观察空间 (Observation Space): {env.observation_space}")
print(f"动作空间 (Action Space): {env.action_space}")
print(f"动作范围: {env.action_space.high}, {env.action_space.low}")

pybullet build time: Oct 21 2025 17:40:50


argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
观察空间 (Observation Space): Dict('achieved_goal': Box(-10.0, 10.0, (3,), float32), 'desired_goal': Box(-10.0, 10.0, (3,), float32), 'observation': Box(-10.0, 10.0, (6,), float32))
动作空间 (Action Space): Box(-1.0, 1.0, (3,), float32)
动作范围: [1. 1. 1.], [-1. -1. -1.]


> 解释：  
观察空间是 Dict：包含 observation, desired_goal 等。  
结论：我必须使用 SB3 的 MultiInputPolicy，普通的 MlpPolicy 处理不了字典。  
动作空间是 Box(3,) 且连续：  
结论：排除 DQN（它是离散的）。我需要在 SAC, PPO, DDPG, TD3 中选。  
任务属性：这是机器人控制（Robotics）。  
结论：SAC 是目前的行业首选，因为它的样本效率最高（Off-policy）。  

In [ ]:
# Random Baseline 探索一下奖励情况
obs, _ = env.reset()
total_reward = 0
for _ in range(1000):
    action = env.action_space.sample() # 随机采样
    obs, reward, terminated, truncated, _ = env.step(action)
    total_reward += reward
    if terminated or truncated:
        obs, _ = env.reset()
print(f"随机策略平均奖励: {total_reward / 1000}")

> 解释：  
如果奖励全是 0，说明是稀疏奖励（极难）。  
如果奖励是连续的小负数，说明是密集奖励（较易）。  
PandaReach 结论：它有 dense 模式，我们可以从简单的 dense 开始。  

### experiment_01 SAC最简化实现

In [4]:
# SAC 最简化实现 & TensorBoard
import os
from stable_baselines3 import SAC

# 1.1. 配置文件 save 路径
exp_id = "experiment_01_random"
algo_name = "SAC"
run_id = "run_01"
base_path = f"./outputs/{exp_id}" # 同一架构不同算法不同超参放可以放一起
sub_path = os.path.join(base_path, f"{algo_name}_{run_id}")
os.makedirs(sub_path, exist_ok=True)
log_dir = os.path.join(sub_path, "logs")   # 给 TensorBoard
os.makedirs(log_dir, exist_ok=True)

# 通过 tensorboard_log 自动把 log 保存到 “tensorboard_log/tb_log_name“ 
model = SAC("MultiInputPolicy", env, verbose=1, tensorboard_log=log_dir)
model.learn(total_timesteps=10000, tb_log_name=f"{run_id}")  # tb_log_name=f"lr_{lr}_{run_id}"

Using cpu device
Logging to ./outputs/experiment_01_random/SAC_run_01/logs/run_01_1
---------------------------------
| rollout/           |          |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 4        |
|    fps             | 387      |
|    time_elapsed    | 0        |
|    total_timesteps | 200      |
| train/             |          |
|    actor_loss      | -4.33    |
|    critic_loss     | 0.13     |
|    ent_coef        | 0.971    |
|    ent_coef_loss   | -0.148   |
|    learning_rate   | 0.0003   |
|    n_updates       | 99       |
---------------------------------
---------------------------------
| rollout/           |          |
|    success_rate    | 0.125    |
| time/              |          |
|    episodes        | 8        |
|    fps             | 293      |
|    time_elapsed    | 1        |
|    total_timesteps | 355      |
| train/             |          |
|    actor_loss      | -5.3     |
|    critic_loss     | 0.0803   

只要你开启了 tensorboard_log，SB3 默认会自动记录以下三类数据（取决于具体算法，SAC 通常包含）：  
rollout/ (评估相关)：  
ep_rew_mean: 过去 100 个 episode 的平均奖励（最重要指标）。  
ep_len_mean: 过去 100 个 episode 的平均步数。  
success_rate: 成功率（如果你的环境提供了 is_success 信息）。  
train/ (训练细节)：  
learning_rate: 当前学习率。  
loss: 总损失。  
actor_loss / critic_loss: 策略和价值网络的各自损失（针对 SAC）。  
ent_coef: 熵系数（反映 SAC 的探索程度）。  
time/ (效率相关)：  
fps: 每秒处理的步数（衡量训练速度）。  
time_elapsed: 训练累计耗时。  


In [ ]:
# 通过 SB3 的 Callback 来记录更多自定义数据
from stable_baselines3.common.callbacks import BaseCallback
'''
class CustomDataCallback(BaseCallback):

my_callback = CustomDataCallback()

model = SAC("MultiInputPolicy", env, tensorboard_log="./logs/")
model.learn(total_timesteps=10000, callback=my_callback)
'''

观察现象：
你会发现奖励（Rew Mean）在动，但提升非常缓慢，或者干脆不动。  
诊断： 检查 TensorBoard，如果你发现输入数据的数值（如坐标）跨度很大（从 0 到 1），而网络权重更新很吃力。  
药方： 必须引入 VecNormalize（归一化）。  

### experiment_02 SAC最简化+DummyVecEnv, VecNormalize

In [ ]:
import os
import gymnasium as gym
import panda_gym
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# 1.1. 配置文件 save 路径
exp_id = "experiment_02_random"
algo_name = "SAC"
run_id = "run_01_random_baseline"
base_path = f"./outputs/{exp_id}" # 同一架构不同算法不同超参放可以放一起
sub_path = os.path.join(base_path, f"{algo_name}_{run_id}")
os.makedirs(sub_path, exist_ok=True)
log_dir = os.path.join(sub_path, "logs")   # 给 TensorBoard
model_path = os.path.join(sub_path, "models") # 给权重文件
os.makedirs(log_dir, exist_ok=True)
os.makedirs(model_path, exist_ok=True)

# 1.2. 常规配置
env_id = "PandaReach-v3"

# 2. 包装环境
def make_env():
    # 当前环境设置 reward_type="dense" 确保快速收敛
    env = gym.make(env_id, reward_type="dense")
    return env
# env = make_env()

# VecEnv 需要在一个 lambda 函数中创建
env = DummyVecEnv([make_env])

# 添加 VecNormalize 包装器
# norm_obs: 是否归一化观测值 (通常建议开启)—稳定Actors-Critic网络
# norm_reward: 是否归一化奖励 (通常建议开启)—稳定Critic网络
# clip_obs: 裁剪观测值的范围—防止极端异常值破坏神经网络的权重
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.)

# 3. main & tensorboard
# 注意：使用 VecNormalize 后，模型输入的是归一化后的数据
model = SAC(
    "MultiInputPolicy", 
    env, 
    verbose=1, 
    tensorboard_log=log_dir
)
model.learn(
    total_timesteps=10000, 
    tb_log_name=run_id
)

# 4. save
model.save(os.path.join(model_path, "sac_model"))  # save：模型参数和结构
env.save(os.path.join(model_path, "vec_normalize.pkl"))  # !: save VecNormalize 过程中的归一化参数(均值、方差等)，以便后续评估或继续训练时使用

#model.save(model_path)
#env.save(f"{model_path}_vec_normalize.pkl")

print(f"训练完成，模型已保存至: {model_path}")

pybullet build time: Oct 21 2025 17:40:50


argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
Using cpu device
Logging to ./outputs/experiment_03_random/SAC_run_01_random_baseline/logs/run_01_random_baseline_1
---------------------------------
| rollout/           |          |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 4        |
|    fps             | 304      |
|    time_elapsed    | 0        |
|    total_timesteps | 200      |
| train/             |          |
|    actor_loss      | -4.5     |
|    critic_loss     | 0.119    |
|    ent_coef        | 0.971    |
|    ent_coef_loss   | -0.149   |
|    learning_rate   | 0.0003   |
|    n_updates       | 99       |
---------------------------------
---------------------------------
| rollout/           |          |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 8        |
|    fps        

In [2]:
import os
import gymnasium as gym
import panda_gym
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback

# 1.1. 配置文件 save 路径
exp_id = "experiment_0F"
algo_name = "SAC"
run_id = "run_01_random_baseline"
base_path = f"./outputs/{exp_id}" # 同一架构不同算法不同超参放可以放一起
sub_path = os.path.join(base_path, f"{algo_name}_{run_id}")
os.makedirs(sub_path, exist_ok=True)
log_dir = os.path.join(sub_path, "logs")   # 给 TensorBoard
model_path = os.path.join(sub_path, "models") # 给权重文件
os.makedirs(log_dir, exist_ok=True)
os.makedirs(model_path, exist_ok=True)

# --- 1. 环境定义与配置 ---
env_id = "PandaReach-v3"

# 自动保存归一化的统计量（这是解决加载模型后变傻的关键）
class SaveVecNormalizeCallback(BaseCallback):
    def __init__(self, save_path: str, verbose=1):
        super(SaveVecNormalizeCallback, self).__init__(verbose)
        self.save_path = save_path
    def _on_step(self) -> bool:
        if self.n_calls % 5000 == 0:
            self.model.get_vec_normalize_env().save(self.save_path)
        return True

# --- 2. 创建环境逻辑 ---
def make_env():
    # 使用 reward_type="dense" 确保快速收敛
    env = gym.make(env_id, reward_type="dense")
    return env

# 包装环境：向量化 + 归一化
env = DummyVecEnv([make_env])
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.)

# 评估环境（不更新归一化参数）
eval_env = DummyVecEnv([make_env])
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, training=False)

# --- 3. 设置 Callbacks ---
# 评估回调：每 2000 步测试一次成功率
eval_callback = EvalCallback(
    eval_env, 
    best_model_save_path=log_dir,
    log_path=log_dir, 
    eval_freq=2000,
    deterministic=True
)
save_vec_callback = SaveVecNormalizeCallback(save_path=os.path.join(log_dir, "vec_normalize.pkl"))

# --- 4. 定义 SAC 算法 ---
model = SAC(
    "MultiInputPolicy", 
    env,
    learning_rate=1e-3,
    buffer_size=100000,
    batch_size=256,
    tau=0.005,
    gamma=0.99,
    tensorboard_log=log_dir,
    verbose=1,
    device="cpu"  # 对于简单的 MLP，Mac 的 CPU 通常比 GPU 效率更高
)

# --- 5. 开始训练 ---
print("开始训练 PandaReach 任务...")
model.learn(
    total_timesteps=50000, 
    callback=[eval_callback, save_vec_callback],
    tb_log_name=run_id
)

# --- 6. 最终保存 ---
model.save(os.path.join(model_path, "sac_panda_reach_final"))
env.save(os.path.join(model_path, "vec_normalize.pkl"))
print(f"训练完成！模型已保存至 {model_path}")

argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
Using cpu device
argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
开始训练 PandaReach 任务...
Logging to ./outputs/experiment_0F/SAC_run_01_random_baseline/logs/run_01_random_baseline_1
---------------------------------
| rollout/           |          |
|    success_rate    | 0.25     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 444      |
|    time_elapsed    | 0        |
|    total_timesteps | 171      |
| train/             |          |
|    actor_loss      | -4.21    |
|    critic_loss     | 0.103    |
|    ent_coef        | 0.933    |
|    ent_coef_loss   | -0.348   |
|    learning_rate   | 0.001    |
|    n_updates       | 70       |
---------------------------------
-----------------

/opt/miniconda3/envs/mujoco/lib/python3.10/site-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


New best mean reward!
---------------------------------
| rollout/           |          |
|    success_rate    | 0.114    |
| time/              |          |
|    episodes        | 44       |
|    fps             | 232      |
|    time_elapsed    | 8        |
|    total_timesteps | 2017     |
| train/             |          |
|    actor_loss      | -6.58    |
|    critic_loss     | 0.0302   |
|    ent_coef        | 0.149    |
|    ent_coef_loss   | -9.41    |
|    learning_rate   | 0.001    |
|    n_updates       | 1916     |
---------------------------------
---------------------------------
| rollout/           |          |
|    success_rate    | 0.125    |
| time/              |          |
|    episodes        | 48       |
|    fps             | 232      |
|    time_elapsed    | 9        |
|    total_timesteps | 2168     |
| train/             |          |
|    actor_loss      | -6.49    |
|    critic_loss     | 0.0215   |
|    ent_coef        | 0.128    |
|    ent_coef_loss   | -10